# Integrated Data Quality and Feature Analysis

## Purpose

This notebook is the **data quality assessment and feature analysis stage** following the integration process.

It starts from the controlled dataset produced by `integration.ipynb` and determines whether the integrated data is reliable and suitable for downstream feature engineering and predictive modelling.

> **Principle: Understand the data before changing the data.**

---

## Input Dataset

The input is:

`london_property_intelligence_integrated_borough_year.parquet`

This dataset was produced by integrating:

* Property
* Population
* Income
* Crime
* PTAL
* IMD

The intended analytical grain is:

> **One row per London Borough-Year**

Current structure:

* **33 London boroughs**
* **2018–2025**
* **264 Borough-Year observations**

---

## Analysis Scope

This notebook will assess:

1. **Structural quality**

   * Shape, data types, duplicates, and grain integrity
   * Borough and temporal coverage

2. **Missing data**

   * Missingness by feature, borough, and year
   * Structural versus unexpected missingness
   * Source-specific temporal coverage

3. **Feature behaviour**

   * Descriptive statistics and distributions
   * Variation, skewness, and potential outliers
   * Feature consistency and usefulness

4. **Feature relationships**

   * Correlations and potential redundancy
   * Multicollinearity risks
   * Relationships between major feature groups

5. **Target and leakage assessment**

   * Review of the target variables
   * Temporal availability of predictors
   * Detection of potential data leakage

6. **Feature readiness**

   * Identify features requiring transformation, investigation, or exclusion
   * Establish which features are suitable for downstream modelling

---

## Boundaries

This notebook **does not rebuild the integration pipeline**.

There should be no unnecessary remapping or merging of the original source datasets here. If an integration problem is discovered, the correction should be made in `integration.ipynb` and the integrated dataset regenerated.

Likewise, modelling is outside the scope of this notebook.

### Pipeline

**Integration → Data Quality → Feature Analysis → Feature Engineering → Modelling**

The outcome of this notebook should be a clear, evidence-based understanding of the quality, limitations, and modelling readiness of the integrated features.


## Data Quality & Feature Review

Data Quality
     ↓
Missingness
     ↓
Temporal Coverage
     ↓
Borough Coverage
     ↓
Outliers
     ↓
Distributions
     ↓
Target Alignment
     ↓
Feature → Target Correlation
     ↓
Correlation Sample Size
     ↓
Feature ↔ Feature Correlation
     ↓
Multicollinearity
     ↓
Feature Selection
     ↓
Modeling

In [1]:
import pandas as pd

integration_input_path = (
    "../data/processed/"
    "london_property_intelligence_integrated_borough_year.parquet"
)

df = pd.read_parquet(integration_input_path)

print("=== INTEGRATED DATASET LOADED ===")
print("Shape:", df.shape)
print("Columns:", len(df.columns))
print("Unique Borough-Year:", df[["Borough_Code", "Year"]].drop_duplicates().shape[0])

=== INTEGRATED DATASET LOADED ===
Shape: (264, 41)
Columns: 41
Unique Borough-Year: 264


In [2]:
print("=== STRUCTURAL QUALITY BASELINE ===")

print("Rows:", len(df))
print("Columns:", len(df.columns))

print(
    "Duplicate rows:",
    df.duplicated().sum()
)

print(
    "Duplicate Borough-Year:",
    df.duplicated(
        subset=["Borough_Code", "Year"]
    ).sum()
)

print(
    "Unique Boroughs:",
    df["Borough_Code"].nunique()
)

print(
    "Year range:",
    df["Year"].min(),
    "to",
    df["Year"].max()
)

=== STRUCTURAL QUALITY BASELINE ===
Rows: 264
Columns: 41
Duplicate rows: 0
Duplicate Borough-Year: 0
Unique Boroughs: 33
Year range: 2018 to 2025


## 2. Feature Inventory

Before assessing missingness and feature behaviour, the variables in the integrated dataset are classified by their analytical role.

This prevents identifiers, temporal variables, predictors, and target variables from being treated as interchangeable features.


In [3]:
print("=== FEATURE INVENTORY ===")

for i, column in enumerate(df.columns, start=1):
    print(
        f"{i:02d}. {column:<50} "
        f"dtype={df[column].dtype}"
    )

=== FEATURE INVENTORY ===
01. District                                           dtype=object
02. Year                                               dtype=int64
03. Transactions                                       dtype=int64
04. Average_Price                                      dtype=float64
05. Median_Price                                       dtype=float64
06. Min_Price                                          dtype=int64
07. Max_Price                                          dtype=int64
08. Price_STD                                          dtype=float64
09. Average_Price_Growth                               dtype=float64
10. Median_Price_Growth                                dtype=float64
11. Target_Average_Price_Growth                        dtype=float64
12. Target_Median_Price_Growth                         dtype=float64
13. Borough_Code                                       dtype=object
14. Population                                         dtype=float64
15. Mean_Income   

## 3. Missingness Profile

Missing values are assessed before any cleaning or imputation is applied.

The purpose is to determine the **amount, distribution, and likely structural cause** of missingness across the integrated features.

Missingness is not automatically treated as a data error. Some gaps are expected because the underlying source datasets have different temporal coverage or because target variables require future observations.

The analysis therefore examines missingness both **overall and by year**.


In [4]:
print("=== MISSINGNESS PROFILE ===")

missing_count = df.isna().sum()
missing_pct = (missing_count / len(df) * 100).round(1)

missing_profile = (
    pd.DataFrame({
        "Missing_Count": missing_count,
        "Missing_%": missing_pct
    })
    .query("Missing_Count > 0")
    .sort_values("Missing_Count", ascending=False)
)

print(missing_profile)

=== MISSINGNESS PROFILE ===
                                            Missing_Count  Missing_%
IMD_Proportion_Most_Deprived_10pct                    231       87.5
IMD_Rank_of_Proportion_Most_Deprived_10pct            231       87.5
IMD_Extent                                            231       87.5
IMD_Rank_of_Extent                                    231       87.5
IMD_Rank_of_Local_Concentration                       231       87.5
IMD_Local_Concentration                               231       87.5
IMD_Average_Score                                     231       87.5
IMD_Rank_of_Average_Score                             231       87.5
IMD_Rank_of_Average_Rank                              231       87.5
IMD_Average_Rank                                      231       87.5
DRUG OFFENCES                                         168       63.6
FRAUD AND FORGERY                                     168       63.6
POSSESSION OF WEAPONS                                 168       63.6
MISCEL

## Understanding Missing Data

The missing values in the integrated dataset do not necessarily represent data errors.

The source datasets have different periods of coverage, and some variables are only available for specific years. The target variables also have missing values where the required future observation is not available.

Therefore, missing values are investigated by **year and by feature** before any decision is made about removing, imputing, or retaining them.

At this stage, no missing values are changed.


## 4. Temporal Coverage of Features

The integrated dataset covers 2018–2025, but not every source dataset covers the same years.

This check shows which years contain data for each feature. It helps distinguish expected gaps caused by source coverage from unexpected missing values.

No values are changed at this stage.


In [5]:
print("=== FEATURE TEMPORAL COVERAGE ===")

coverage_rows = []

for column in df.columns:
    if column in ["District", "Borough_Code", "Year"]:
        continue

    available_by_year = (
        df.groupby("Year")[column]
        .apply(lambda x: x.notna().sum())
    )

    years_with_data = available_by_year[
        available_by_year > 0
    ].index.tolist()

    coverage_rows.append({
        "Feature": column,
        "Years_with_Data": len(years_with_data),
        "First_Year": min(years_with_data) if years_with_data else None,
        "Last_Year": max(years_with_data) if years_with_data else None,
        "Years": years_with_data
    })

temporal_coverage = pd.DataFrame(coverage_rows)

print(temporal_coverage.to_string(index=False))

=== FEATURE TEMPORAL COVERAGE ===
                                   Feature  Years_with_Data  First_Year  Last_Year                                            Years
                              Transactions                8        2018       2025 [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
                             Average_Price                8        2018       2025 [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
                              Median_Price                8        2018       2025 [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
                                 Min_Price                8        2018       2025 [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
                                 Max_Price                8        2018       2025 [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
                                 Price_STD                8        2018       2025 [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
                      Average_Price_Growth

## 5. Coverage Completeness

Temporal coverage tells us **which years** a feature contains data.

The next check asks a more specific question:

> When a source is expected to have data for a particular year, does it actually cover all 33 boroughs?

This helps distinguish a complete source-level coverage pattern from individual missing borough observations.

No values are changed at this stage.


In [6]:
print("=== COVERAGE COMPLETENESS ===")

coverage_rows = []

feature_columns = [
    column
    for column in df.columns
    if column not in ["District", "Year", "Borough_Code"]
]

for column in feature_columns:

    yearly_counts = (
        df.groupby("Year")[column]
        .apply(lambda x: x.notna().sum())
    )

    for year, count in yearly_counts.items():

        if count > 0:
            coverage_rows.append({
                "Feature": column,
                "Year": year,
                "Boroughs_with_Data": count,
                "Expected_Boroughs": 33,
                "Complete": count == 33
            })

coverage_completeness = pd.DataFrame(coverage_rows)

print(
    coverage_completeness[
        ~coverage_completeness["Complete"]
    ].to_string(index=False)
)

=== COVERAGE COMPLETENESS ===
                             Feature  Year  Boroughs_with_Data  Expected_Boroughs  Complete
           ARSON AND CRIMINAL DAMAGE  2021                  32                 33     False
           ARSON AND CRIMINAL DAMAGE  2022                  32                 33     False
           ARSON AND CRIMINAL DAMAGE  2023                  32                 33     False
                            BURGLARY  2021                  32                 33     False
                            BURGLARY  2022                  32                 33     False
                            BURGLARY  2023                  32                 33     False
                       DRUG OFFENCES  2021                  32                 33     False
                       DRUG OFFENCES  2022                  32                 33     False
                       DRUG OFFENCES  2023                  32                 33     False
                   FRAUD AND FORGERY  2021        

## 6. Identifying Incomplete Borough Coverage

The Crime features have data for 32 of the 33 boroughs in each available year.

Before deciding how to handle this gap, we identify the missing borough and verify whether the absence is consistent across the Crime dataset.

No values are changed at this stage.


In [7]:
print("=== INCOMPLETE CRIME BOROUGH COVERAGE ===")

crime_features = [
    column
    for column in df.columns
    if column in [
        "ARSON AND CRIMINAL DAMAGE",
        "BURGLARY",
        "DRUG OFFENCES",
        "FRAUD AND FORGERY",
        "MISCELLANEOUS CRIMES AGAINST SOCIETY",
        "POSSESSION OF WEAPONS",
        "PUBLIC ORDER OFFENCES",
        "ROBBERY",
        "SEXUAL OFFENCES",
        "THEFT",
        "VEHICLE OFFENCES",
        "VIOLENCE AGAINST THE PERSON",
    ]
]

crime_missing_by_year = {}

for year in [2021, 2022, 2023]:
    missing_boroughs = df.loc[
        df["Year"].eq(year)
        & df[crime_features].isna().all(axis=1),
        "Borough_Code"
    ].tolist()

    crime_missing_by_year[year] = missing_boroughs

for year, boroughs in crime_missing_by_year.items():
    print(f"{year}: {boroughs}")

=== INCOMPLETE CRIME BOROUGH COVERAGE ===
2021: ['E09000001']
2022: ['E09000001']
2023: ['E09000001']


In [8]:
print("=== MISSINGNESS AND COVERAGE SUMMARY ===")

summary_rows = []

for column in df.columns:
    if column in ["District", "Borough_Code", "Year"]:
        continue

    missing_count = df[column].isna().sum()
    missing_pct = round(missing_count / len(df) * 100, 1)

    yearly_counts = (
        df.groupby("Year")[column]
        .apply(lambda x: x.notna().sum())
    )

    years_with_data = yearly_counts[yearly_counts > 0]

    summary_rows.append({
        "Feature": column,
        "Missing_%": missing_pct,
        "Years_with_Data": len(years_with_data),
        "First_Year": years_with_data.index.min(),
        "Last_Year": years_with_data.index.max(),
        "Minimum_Borough_Coverage": years_with_data.min(),
        "Maximum_Borough_Coverage": years_with_data.max()
    })

missing_coverage_summary = (
    pd.DataFrame(summary_rows)
    .sort_values("Missing_%", ascending=False)
)

print(
    missing_coverage_summary.to_string(index=False)
)

=== MISSINGNESS AND COVERAGE SUMMARY ===
                                   Feature  Missing_%  Years_with_Data  First_Year  Last_Year  Minimum_Borough_Coverage  Maximum_Borough_Coverage
                        IMD_Rank_of_Extent       87.5                1        2019       2019                        33                        33
                                IMD_Extent       87.5                1        2019       2019                        33                        33
IMD_Rank_of_Proportion_Most_Deprived_10pct       87.5                1        2019       2019                        33                        33
        IMD_Proportion_Most_Deprived_10pct       87.5                1        2019       2019                        33                        33
                 IMD_Rank_of_Average_Score       87.5                1        2019       2019                        33                        33
                         IMD_Average_Score       87.5                1        2019 

## Missingness Findings

The missingness patterns are largely explained by the temporal coverage of the underlying source datasets and by the construction of growth and target variables.

Key findings:

* Property features have complete coverage across 2018–2025.
* Population covers 2018–2022 with all 33 boroughs represented.
* Income covers its available source period.
* Crime covers 2021–2023, with City of London absent from the source data.
* IMD is available for 2019 only, with all 33 boroughs represented.
* Growth and target variables have expected boundary-year missingness caused by their calculation requirements.

These patterns are considered **structural missingness rather than automatic data errors**.

No missing values are imputed or removed at this stage. Decisions about feature use will be made later based on the modelling objective and the information available at each prediction point.


## 6. Descriptive Statistics

We now examine the numerical features to understand their scale, variation, and distribution before applying any transformations or feature engineering.

This is an exploratory step. No values are changed at this stage.


In [9]:
print("=== DESCRIPTIVE STATISTICS ===")

numeric_features = df.select_dtypes(
    include="number"
).columns.tolist()

descriptive_stats = (
    df[numeric_features]
    .describe()
    .T
)

descriptive_stats["missing"] = (
    df[numeric_features].isna().sum()
)

print(descriptive_stats)

=== DESCRIPTIVE STATISTICS ===
                                            count          mean           std  \
Year                                        264.0  2.021500e+03  2.295640e+00   
Transactions                                264.0  3.360716e+03  1.179154e+03   
Average_Price                               264.0  9.698969e+05  1.120579e+06   
Median_Price                                264.0  5.530846e+05  1.893004e+05   
Min_Price                                   264.0  8.817197e+02  2.760079e+03   
Max_Price                                   264.0  9.403684e+07  1.105229e+08   
Price_STD                                   264.0  2.929790e+06  4.281858e+06   
Average_Price_Growth                        231.0  1.032891e-02  1.864954e-01   
Median_Price_Growth                         231.0  1.400544e-02  4.573977e-02   
Target_Average_Price_Growth                 231.0  1.032891e-02  1.864954e-01   
Target_Median_Price_Growth                  231.0  1.400544e-02  4.573977e-02 

## 7. Outlier Screening

The descriptive statistics highlight features with large ranges or substantial variation.

We now perform an initial statistical screening for potential outliers using the IQR method.

This is only a screening step. A statistically unusual value is not automatically an error and will not be removed without further investigation.


In [10]:
print("=== OUTLIER SCREENING (IQR) ===")

outlier_summary = []

for column in numeric_features:

    series = df[column].dropna()

    if series.empty:
        continue

    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = series[
        (series < lower_bound) |
        (series > upper_bound)
    ]

    outlier_summary.append({
        "Feature": column,
        "Outlier_Count": len(outliers),
        "Outlier_%": round(
            len(outliers) / len(series) * 100, 1
        ),
        "Lower_Bound": lower_bound,
        "Upper_Bound": upper_bound
    })

outlier_summary = (
    pd.DataFrame(outlier_summary)
    .sort_values("Outlier_Count", ascending=False)
)

print(outlier_summary.to_string(index=False))

=== OUTLIER SCREENING (IQR) ===
                                   Feature  Outlier_Count  Outlier_%   Lower_Bound  Upper_Bound
                             Average_Price             33       12.5  1.377138e+04 1.406207e+06
                                 Price_STD             31       11.7 -2.385400e+06 6.402511e+06
                                 Max_Price             27       10.2 -8.373392e+07 2.212788e+08
                              Median_Price             25        9.5  2.110312e+05 8.142812e+05
                               Mean_Income             22       11.1  8.050000e+03 9.705000e+04
                                AvPTAI2015             16        6.1 -1.236232e+01 3.538341e+01
                      Average_Price_Growth             14        6.1 -2.396752e-01 2.397216e-01
                                 Min_Price             14        5.3 -1.250000e+03 2.350000e+03
                              Transactions             14        5.3  6.598750e+02 5.942875e+03
        

### What the Outlier Check Shows

The first outlier check shows that some of the property features have a relatively high number of values outside the IQR range. `Average_Price` has 33 such values, `Price_STD` has 31, `Max_Price` has 27, and `Median_Price` has 25.

`Mean_Income` also has 22 values flagged as outliers, while `AvPTAI2015` has 16.

These results do not mean that these values are wrong. London boroughs can differ substantially in property prices, income, and other characteristics, so some extreme values may be genuine.

One result that needs closer attention is `FRAUD AND FORGERY`, where the IQR range is zero and 11 values are flagged. This suggests that the feature has a very concentrated distribution and should be looked at more closely.

For now, no values are removed or changed. The flagged observations will only be investigated further where necessary.


## 8. Investigating Property Price Outliers

The IQR screening flagged several property price features, especially `Average_Price`, `Median_Price`, `Max_Price`, and `Price_STD`.

Before deciding whether any of these values need treatment, we check where the flagged observations occur by borough and year.

No values are changed at this stage.


In [11]:
print("=== PROPERTY PRICE OUTLIER LOCATIONS ===")

property_features = [
    "Average_Price",
    "Median_Price",
    "Max_Price",
    "Price_STD"
]

for column in property_features:

    series = df[column]

    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    mask = (
        (series < lower_bound) |
        (series > upper_bound)
    )

    outliers = df.loc[
        mask,
        ["District", "Borough_Code", "Year", column]
    ].sort_values(column, ascending=False)

    print(f"\n--- {column} ---")
    print("Outlier count:", len(outliers))
    print(outliers.to_string(index=False))

=== PROPERTY PRICE OUTLIER LOCATIONS ===

--- Average_Price ---
Outlier count: 33
              District Borough_Code  Year  Average_Price
        CITY OF LONDON    E09000001  2019   1.450477e+07
        CITY OF LONDON    E09000001  2023   5.480314e+06
        CITY OF LONDON    E09000001  2018   4.905095e+06
        CITY OF LONDON    E09000001  2021   3.989396e+06
        CITY OF LONDON    E09000001  2022   3.697610e+06
   CITY OF WESTMINSTER    E09000033  2018   3.484257e+06
        CITY OF LONDON    E09000001  2024   3.426844e+06
   CITY OF WESTMINSTER    E09000033  2020   3.160633e+06
   CITY OF WESTMINSTER    E09000033  2023   3.106188e+06
   CITY OF WESTMINSTER    E09000033  2022   2.956530e+06
   CITY OF WESTMINSTER    E09000033  2024   2.933762e+06
   CITY OF WESTMINSTER    E09000033  2019   2.867918e+06
   CITY OF WESTMINSTER    E09000033  2021   2.831910e+06
        CITY OF LONDON    E09000001  2020   2.755823e+06
KENSINGTON AND CHELSEA    E09000020  2022   2.755059e+06
KENSIN

In [12]:
print("=== AVERAGE PRICE OUTLIERS BY BOROUGH ===")

print(
    outliers.groupby("District")
    .size()
    .sort_values(ascending=False)
)

=== AVERAGE PRICE OUTLIERS BY BOROUGH ===
District
CITY OF LONDON            8
CITY OF WESTMINSTER       8
CAMDEN                    4
ISLINGTON                 2
TOWER HAMLETS             2
GREENWICH                 1
KENSINGTON AND CHELSEA    1
HAMMERSMITH AND FULHAM    1
NEWHAM                    1
SOUTHWARK                 1
WALTHAM FOREST            1
WANDSWORTH                1
dtype: int64


In [13]:
print("=== AVERAGE PRICE OUTLIERS BY BOROUGH ===")

column = "Average_Price"

Q1 = df[column].quantile(0.25)
Q3 = df[column].quantile(0.75)
IQR = Q3 - Q1

mask = (
    (df[column] < Q1 - 1.5 * IQR) |
    (df[column] > Q3 + 1.5 * IQR)
)

average_price_outliers = df.loc[
    mask,
    ["District", "Borough_Code", "Year", "Average_Price"]
]

print(
    average_price_outliers
    .groupby(["District", "Borough_Code"])
    .size()
    .sort_values(ascending=False)
)

=== AVERAGE PRICE OUTLIERS BY BOROUGH ===
District                Borough_Code
CITY OF LONDON          E09000001       8
CITY OF WESTMINSTER     E09000033       8
KENSINGTON AND CHELSEA  E09000020       8
CAMDEN                  E09000007       7
HAMMERSMITH AND FULHAM  E09000013       1
TOWER HAMLETS           E09000030       1
dtype: int64


## 9. Looking at Feature Distributions

The outlier check showed that some features contain unusually high or low values.

Before moving to correlations, we look at the median and key quantiles to understand how the values are distributed. This helps us see whether extreme values are isolated or part of the overall pattern.

No values are changed at this stage.


In [14]:
print("=== FEATURE DISTRIBUTION SUMMARY ===")

distribution_summary = (
    df[numeric_features]
    .describe(percentiles=[0.25, 0.50, 0.75])
    .T[
        ["count", "25%", "50%", "75%", "mean", "std", "min", "max"]
    ]
)

print(distribution_summary.to_string())

=== FEATURE DISTRIBUTION SUMMARY ===
                                            count           25%           50%           75%          mean           std           min           max
Year                                        264.0  2.019750e+03  2.021500e+03  2.023250e+03  2.021500e+03  2.295640e+00  2.018000e+03  2.025000e+03
Transactions                                264.0  2.641000e+03  3.196000e+03  3.961750e+03  3.360716e+03  1.179154e+03  2.210000e+02  7.470000e+03
Average_Price                               264.0  5.359348e+05  6.951213e+05  8.840438e+05  9.698969e+05  1.120579e+06  3.714191e+05  1.450477e+07
Median_Price                                264.0  4.372500e+05  5.000000e+05  5.880625e+05  5.530846e+05  1.893004e+05  3.067500e+05  1.353777e+06
Min_Price                                   264.0  1.000000e+02  5.000000e+02  1.000000e+03  8.817197e+02  2.760079e+03  1.000000e+00  3.000000e+04
Max_Price                                   264.0  3.064583e+07  5.500000e+

## 10. Checking Target Alignment

The target variables appear to have the same distribution as the current-year growth variables.

Before continuing with the analysis, we check whether the target values correctly represent the following year's price growth.

No values are changed at this stage.


In [15]:
print("=== TARGET ALIGNMENT CHECK ===")

check = (
    df[
        [
            "District",
            "Year",
            "Average_Price_Growth",
            "Target_Average_Price_Growth",
            "Median_Price_Growth",
            "Target_Median_Price_Growth"
        ]
    ]
    .sort_values(["District", "Year"])
)

print(check.head(20).to_string(index=False))

=== TARGET ALIGNMENT CHECK ===
            District  Year  Average_Price_Growth  Target_Average_Price_Growth  Median_Price_Growth  Target_Median_Price_Growth
BARKING AND DAGENHAM  2018                   NaN                     0.059790                  NaN                    0.010595
BARKING AND DAGENHAM  2019              0.059790                     0.189852             0.010595                    0.032258
BARKING AND DAGENHAM  2020              0.189852                    -0.099888             0.032258                    0.046875
BARKING AND DAGENHAM  2021             -0.099888                     0.192617             0.046875                    0.104478
BARKING AND DAGENHAM  2022              0.192617                    -0.129364             0.104478                    0.010811
BARKING AND DAGENHAM  2023             -0.129364                    -0.070744             0.010811                   -0.037433
BARKING AND DAGENHAM  2024             -0.070744                    -0.019078   

### Target Alignment Confirmed

The target variables are correctly aligned with the following year's price growth.

For each borough, the target for year *t* corresponds to the observed price growth in year *t + 1*. The missing targets in 2025 are therefore expected because the following year's growth is not available in the dataset.

No target values were changed.


## 11. Correlation Analysis

We now examine the relationships between the numerical features and the property price growth variables.

The purpose is to identify potentially useful relationships and understand how the features move with price growth.

Correlation does not show causation, and no features are removed or changed based on correlation alone.


In [16]:
print("=== CORRELATION WITH TARGETS ===")

correlation_targets = [
    "Target_Average_Price_Growth",
    "Target_Median_Price_Growth"
]

numeric_for_correlation = df.select_dtypes(
    include="number"
).columns.tolist()

numeric_for_correlation = [
    col for col in numeric_for_correlation
    if col not in [
        "Year",
        "Target_Average_Price_Growth",
        "Target_Median_Price_Growth"
    ]
]

correlation_results = (
    df[numeric_for_correlation + correlation_targets]
    .corr()[correlation_targets]
    .drop(correlation_targets)
    .sort_values(
        "Target_Average_Price_Growth",
        ascending=False
    )
)

print(correlation_results.to_string())

=== CORRELATION WITH TARGETS ===
                                            Target_Average_Price_Growth  Target_Median_Price_Growth
IMD_Average_Rank                                               0.314206                   -0.063749
IMD_Average_Score                                              0.313092                   -0.089630
IMD_Extent                                                     0.301598                   -0.133485
IMD_Local_Concentration                                        0.196137                   -0.076897
AvPTAI2015                                                     0.105074                   -0.185469
Median_Price_Growth                                            0.060806                   -0.042461
Median_Income                                                  0.048213                   -0.185932
IMD_Proportion_Most_Deprived_10pct                             0.038537                   -0.285206
Mean_Income                                                    0.03

### What the Correlation Results Show

The strongest relationship with the target is between current `Average_Price_Growth` and the following year's `Target_Average_Price_Growth` (correlation = -0.54), indicating a moderate negative linear relationship in this dataset.

Some IMD variables also show moderate correlations with the target, while most crime, income, population, and transaction variables show relatively weak linear relationships.

These results are used to understand the data and identify relationships worth investigating further. They are not used on their own to remove or select features, and correlation is not interpreted as causation.

The number of available observations also needs to be considered because some features, particularly IMD variables, have limited temporal coverage.


### Checking Correlation Sample Size

Correlation values are more useful when we also know how many observations were available for each feature.

This check shows the number of non-missing observations used for each feature and target. It helps us distinguish relationships supported by substantial data from those based on limited coverage.

No features or values are changed at this stage.


In [17]:
print("=== CORRELATION SAMPLE SIZE ===")

target_columns = [
    "Target_Average_Price_Growth",
    "Target_Median_Price_Growth"
]

features_for_correlation = [
    col for col in df.select_dtypes(include="number").columns
    if col not in [
        "Year",
        "Target_Average_Price_Growth",
        "Target_Median_Price_Growth"
    ]
]

sample_size_results = []

for feature in features_for_correlation:
    for target in target_columns:
        valid_count = df[[feature, target]].dropna().shape[0]

        sample_size_results.append({
            "Feature": feature,
            "Target": target,
            "Valid_Observations": valid_count
        })

sample_size_df = pd.DataFrame(sample_size_results)

print(sample_size_df.to_string(index=False))

=== CORRELATION SAMPLE SIZE ===
                                   Feature                      Target  Valid_Observations
                              Transactions Target_Average_Price_Growth                 231
                              Transactions  Target_Median_Price_Growth                 231
                             Average_Price Target_Average_Price_Growth                 231
                             Average_Price  Target_Median_Price_Growth                 231
                              Median_Price Target_Average_Price_Growth                 231
                              Median_Price  Target_Median_Price_Growth                 231
                                 Min_Price Target_Average_Price_Growth                 231
                                 Min_Price  Target_Median_Price_Growth                 231
                                 Max_Price Target_Average_Price_Growth                 231
                                 Max_Price  Target_Median_

## 12. Feature Correlation and Multicollinearity

We now examine the relationships between the features themselves.

The aim is to identify features that contain very similar information and may create multicollinearity problems in some models.

This is a diagnostic step only. No features are removed based on correlation alone.


In [18]:
import numpy as np

print("=== FEATURE-TO-FEATURE CORRELATION ===")

feature_columns = [
    col
    for col in df.select_dtypes(include="number").columns
    if col not in [
        "Year",
        "Target_Average_Price_Growth",
        "Target_Median_Price_Growth"
    ]
]

feature_corr = df[feature_columns].corr()

# Keep only the upper triangle and remove the diagonal
upper_triangle = feature_corr.where(
    np.triu(np.ones(feature_corr.shape), k=1).astype(bool)
)

# Convert to a readable table
feature_corr_pairs = (
    upper_triangle
    .stack()
    .reset_index()
)

feature_corr_pairs.columns = [
    "Feature_1",
    "Feature_2",
    "Correlation"
]

# Sort by absolute correlation
feature_corr_pairs["Absolute_Correlation"] = (
    feature_corr_pairs["Correlation"].abs()
)

feature_corr_pairs = feature_corr_pairs.sort_values(
    "Absolute_Correlation",
    ascending=False
)

print(
    feature_corr_pairs[
        [
            "Feature_1",
            "Feature_2",
            "Correlation"
        ]
    ].to_string(index=False)
)

=== FEATURE-TO-FEATURE CORRELATION ===
                                 Feature_1                                  Feature_2  Correlation
                          IMD_Average_Rank                   IMD_Rank_of_Average_Rank    -0.995019
                  IMD_Rank_of_Average_Rank                  IMD_Rank_of_Average_Score     0.994384
                          IMD_Average_Rank                  IMD_Rank_of_Average_Score    -0.994085
                          IMD_Average_Rank                          IMD_Average_Score     0.993951
                         IMD_Average_Score                  IMD_Rank_of_Average_Score    -0.990266
                   IMD_Local_Concentration            IMD_Rank_of_Local_Concentration    -0.989101
                  IMD_Rank_of_Average_Rank                          IMD_Average_Score    -0.983601
        IMD_Proportion_Most_Deprived_10pct IMD_Rank_of_Proportion_Most_Deprived_10pct    -0.982589
                         IMD_Average_Score                         IMD

## High Feature Correlations

The feature correlation check shows several very strong relationships between some variables.

These relationships are expected in some cases, particularly where variables represent the same underlying measure in different forms, such as IMD scores and their corresponding ranks.

Other strong relationships may indicate overlapping information between features and could become relevant when selecting a modelling approach.

No features are removed at this stage. The strongest correlations are reviewed separately before making any modelling decisions.


In [19]:
print("=== HIGH FEATURE CORRELATIONS (|r| >= 0.80) ===")

high_feature_corr = feature_corr_pairs[
    feature_corr_pairs["Absolute_Correlation"] >= 0.80
].copy()

print(
    high_feature_corr[
        [
            "Feature_1",
            "Feature_2",
            "Correlation"
        ]
    ].to_string(index=False)
)

=== HIGH FEATURE CORRELATIONS (|r| >= 0.80) ===
                         Feature_1                                  Feature_2  Correlation
                  IMD_Average_Rank                   IMD_Rank_of_Average_Rank    -0.995019
          IMD_Rank_of_Average_Rank                  IMD_Rank_of_Average_Score     0.994384
                  IMD_Average_Rank                  IMD_Rank_of_Average_Score    -0.994085
                  IMD_Average_Rank                          IMD_Average_Score     0.993951
                 IMD_Average_Score                  IMD_Rank_of_Average_Score    -0.990266
           IMD_Local_Concentration            IMD_Rank_of_Local_Concentration    -0.989101
          IMD_Rank_of_Average_Rank                          IMD_Average_Score    -0.983601
IMD_Proportion_Most_Deprived_10pct IMD_Rank_of_Proportion_Most_Deprived_10pct    -0.982589
                 IMD_Average_Score                         IMD_Rank_of_Extent    -0.965448
                        Population        

## 13. Multicollinearity Check

The strong feature correlations identified above are reviewed further using VIF.

Because several source datasets have substantial missing values, VIF is first assessed on the relatively complete core features. Features with limited coverage, such as Crime and IMD variables, are not included in this first VIF check.

This does not remove or discard those features. Their usefulness will be assessed separately when the modelling dataset and missing-data strategy are defined.


In [21]:
import numpy as np
import pandas as pd

print("=== VIF CHECK — CORE FEATURES ===")

# Select relatively complete numerical features
vif_candidates = [
    col
    for col in df.select_dtypes(include="number").columns
    if col not in [
        "Year",
        "Target_Average_Price_Growth",
        "Target_Median_Price_Growth"
    ]
    and df[col].notna().mean() >= 0.75
]

print("Features included:")
for col in vif_candidates:
    print("-", col)

# Temporary copy for the VIF calculation
vif_data = df[vif_candidates].copy()

# Median imputation is used only for this diagnostic
vif_data = vif_data.fillna(vif_data.median())

# Add intercept
X = np.column_stack([
    np.ones(len(vif_data)),
    vif_data.values
])

vif_results = []

for i, feature in enumerate(vif_candidates, start=1):
    y = X[:, i]
    
    # All other features
    X_other = np.delete(X, i, axis=1)
    
    # Solve linear regression: y = X_other * beta
    beta = np.linalg.lstsq(
        X_other,
        y,
        rcond=None
    )[0]
    
    y_pred = X_other @ beta
    
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    
    r_squared = 1 - (ss_res / ss_tot)
    
    vif = 1 / (1 - r_squared)
    
    vif_results.append({
        "Feature": feature,
        "VIF": vif
    })

vif_results = pd.DataFrame(vif_results)

vif_results = vif_results.sort_values(
    "VIF",
    ascending=False
)

print("\n=== VIF RESULTS ===")
print(vif_results.to_string(index=False))

=== VIF CHECK — CORE FEATURES ===
Features included:
- Transactions
- Average_Price
- Median_Price
- Min_Price
- Max_Price
- Price_STD
- Average_Price_Growth
- Median_Price_Growth
- Mean_Income
- Median_Income
- Number_of_Individuals
- AvPTAI2015

=== VIF RESULTS ===
              Feature       VIF
            Price_STD 76.882665
        Average_Price 52.044643
            Max_Price 13.897035
         Median_Price  8.379969
          Mean_Income  7.141355
           AvPTAI2015  6.127328
        Median_Income  4.430942
 Average_Price_Growth  2.979917
Number_of_Individuals  2.828917
         Transactions  2.271321
            Min_Price  1.593127
  Median_Price_Growth  1.312415


In [22]:
print("=== VIF CHECK — WITHOUT Price_STD ===")

vif_candidates_reduced = [
    col
    for col in vif_candidates
    if col != "Price_STD"
]

vif_data_reduced = df[vif_candidates_reduced].copy()

# Temporary imputation for diagnostic purposes only
vif_data_reduced = vif_data_reduced.fillna(
    vif_data_reduced.median()
)

X = np.column_stack([
    np.ones(len(vif_data_reduced)),
    vif_data_reduced.values
])

vif_results_reduced = []

for i, feature in enumerate(vif_candidates_reduced, start=1):
    y = X[:, i]

    X_other = np.delete(X, i, axis=1)

    beta = np.linalg.lstsq(
        X_other,
        y,
        rcond=None
    )[0]

    y_pred = X_other @ beta

    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)

    r_squared = 1 - (ss_res / ss_tot)

    vif = 1 / (1 - r_squared)

    vif_results_reduced.append({
        "Feature": feature,
        "VIF": vif
    })

vif_results_reduced = pd.DataFrame(
    vif_results_reduced
).sort_values(
    "VIF",
    ascending=False
)

print("\n=== VIF RESULTS WITHOUT Price_STD ===")
print(vif_results_reduced.to_string(index=False))

=== VIF CHECK — WITHOUT Price_STD ===

=== VIF RESULTS WITHOUT Price_STD ===
              Feature      VIF
        Average_Price 7.190442
          Mean_Income 7.121984
           AvPTAI2015 5.391576
         Median_Price 5.040522
        Median_Income 4.430229
 Average_Price_Growth 2.913516
Number_of_Individuals 2.816643
         Transactions 2.184161
            Min_Price 1.589867
            Max_Price 1.442700
  Median_Price_Growth 1.310198


## VIF Findings

The first VIF check showed very high multicollinearity, mainly involving `Price_STD` and `Average_Price`.

After temporarily removing `Price_STD`, the VIF of `Average_Price` dropped from **52.04 to 7.19**, while `Max_Price` dropped from **13.90 to 1.44**.

This suggests that `Price_STD` is a major source of the multicollinearity among the property price features.

For now, `Price_STD` is flagged as a **candidate for removal during final feature selection**. No feature has been removed from the integrated dataset at this stage.


## Feature Redundancy Review

The correlation and VIF checks identified several groups of features that carry closely related information.

We now review these features from a practical and conceptual perspective. The aim is to identify variables that may be redundant, while keeping potentially useful information available for the later modelling stage.

No features are removed during this review. Any possible removals are recorded as candidates for the final feature-selection stage.


## Population Feature Review

`Population` and `Number_of_Individuals` are very strongly correlated in the integrated dataset.

We compare their coverage and values before deciding whether both variables provide useful and distinct information.

No feature is removed at this stage.


In [23]:
print("=== POPULATION FEATURE REVIEW ===")

population_review = pd.DataFrame({
    "Feature": [
        "Population",
        "Number_of_Individuals"
    ],
    "Non_Missing": [
        df["Population"].notna().sum(),
        df["Number_of_Individuals"].notna().sum()
    ],
    "Missing": [
        df["Population"].isna().sum(),
        df["Number_of_Individuals"].isna().sum()
    ],
    "Coverage_%": [
        df["Population"].notna().mean() * 100,
        df["Number_of_Individuals"].notna().mean() * 100
    ],
    "Mean": [
        df["Population"].mean(),
        df["Number_of_Individuals"].mean()
    ],
    "Median": [
        df["Population"].median(),
        df["Number_of_Individuals"].median()
    ]
})

print(population_review.to_string(index=False))

print("\nCorrelation:")
print(
    df[["Population", "Number_of_Individuals"]]
    .corr()
    .iloc[0, 1]
)

=== POPULATION FEATURE REVIEW ===
              Feature  Non_Missing  Missing  Coverage_%          Mean   Median
           Population          165       99        62.5 268266.048485 279527.0
Number_of_Individuals          198       66        75.0 133217.171717 136000.0

Correlation:
0.9638707195654201


### Population Feature Decision

`Population` and `Number_of_Individuals` are very strongly correlated (`r = 0.964`).

`Number_of_Individuals` also has better coverage (75%) than `Population` (62.5%). Since the two features provide closely related information, `Population` is currently considered a **candidate for removal** in the final feature-selection stage.

No feature is removed from the integrated dataset at this stage.


## Income Feature Review

`Mean_Income` and `Median_Income` are strongly correlated, but they represent different aspects of income.

We compare their coverage and distributions before deciding whether both features should be retained for modelling.


In [24]:
print("=== INCOME FEATURE REVIEW ===")

income_review = pd.DataFrame({
    "Feature": [
        "Mean_Income",
        "Median_Income"
    ],
    "Non_Missing": [
        df["Mean_Income"].notna().sum(),
        df["Median_Income"].notna().sum()
    ],
    "Missing": [
        df["Mean_Income"].isna().sum(),
        df["Median_Income"].isna().sum()
    ],
    "Coverage_%": [
        df["Mean_Income"].notna().mean() * 100,
        df["Median_Income"].notna().mean() * 100
    ],
    "Mean": [
        df["Mean_Income"].mean(),
        df["Median_Income"].mean()
    ],
    "Median": [
        df["Mean_Income"].median(),
        df["Median_Income"].median()
    ]
})

print(income_review.to_string(index=False))

print("\nCorrelation:")
print(
    df[["Mean_Income", "Median_Income"]]
    .corr()
    .iloc[0, 1]
)

=== INCOME FEATURE REVIEW ===
      Feature  Non_Missing  Missing  Coverage_%         Mean  Median
  Mean_Income          198       66        75.0 61880.808081 49000.0
Median_Income          198       66        75.0 34145.959596 32650.0

Correlation:
0.8216443079945167


### Income Feature Decision

`Mean_Income` and `Median_Income` have the same coverage (75%) and a strong correlation (`r = 0.822`).

However, they capture different aspects of income distribution. `Median_Income` is less affected by unusually high incomes, while `Mean_Income` reflects the overall average level.

Both features are therefore **retained for further modelling review** at this stage.


In [26]:
# Review of the IMD feature group

imd_features = [
    "IMD_Average_Score",
    "IMD_Average_Rank",
    "IMD_Rank_of_Average_Score",
    "IMD_Rank_of_Average_Rank",
    "IMD_Extent",
    "IMD_Rank_of_Extent",
    "IMD_Local_Concentration",
    "IMD_Rank_of_Local_Concentration",
    "IMD_Proportion_Most_Deprived_10pct",
    "IMD_Rank_of_Proportion_Most_Deprived_10pct"
]

imd_review = pd.DataFrame({
    "Feature": imd_features,
    "Non_Missing": [df[col].notna().sum() for col in imd_features],
    "Missing": [df[col].isna().sum() for col in imd_features],
    "Coverage_%": [
        round(df[col].notna().mean() * 100, 1)
        for col in imd_features
    ],
    "Mean": [df[col].mean() for col in imd_features],
    "Median": [df[col].median() for col in imd_features]
})

print("=== IMD FEATURE REVIEW ===")
display(imd_review)

=== IMD FEATURE REVIEW ===


,Feature,Non_Missing,Missing,Coverage_%,Mean,Median
0,IMD_Average_Score,33,231,12.5,21.294667,21.5260
1,IMD_Average_Rank,33,231,12.5,17514.279091,18371.2300
2,IMD_Rank_of_Average_Score,33,231,12.5,131.545455,121.0000
3,IMD_Rank_of_Average_Rank,33,231,12.5,118.181818,102.0000
4,IMD_Extent,33,231,12.5,0.162491,0.1630
5,IMD_Rank_of_Extent,33,231,12.5,137.909091,127.0000
6,IMD_Local_Concentration,33,231,12.5,27563.435455,28576.6500
7,IMD_Rank_of_Local_Concentration,33,231,12.5,164.757576,152.0000
8,IMD_Proportion_Most_Deprived_10pct,33,231,12.5,0.021976,0.0083
9,IMD_Rank_of_Proportion_Most_Deprived_10pct,33,231,12.5,166.272727,188.0000


In [27]:
imd_corr = df[imd_features].corr()

print("=== IMD FEATURE CORRELATION ===")
display(imd_corr.round(3))

=== IMD FEATURE CORRELATION ===


,IMD_Average_Score,IMD_Average_Rank,IMD_Rank_of_Average_Score,IMD_Rank_of_Average_Rank,IMD_Extent,IMD_Rank_of_Extent,IMD_Local_Concentration,IMD_Rank_of_Local_Concentration,IMD_Proportion_Most_Deprived_10pct,IMD_Rank_of_Proportion_Most_Deprived_10pct
IMD_Average_Score,1.000,0.994,-0.990,-0.984,0.949,-0.965,0.829,-0.855,0.643,-0.696
IMD_Average_Rank,0.994,1.000,-0.994,-0.995,0.909,-0.944,0.827,-0.843,0.588,-0.645
IMD_Rank_of_Average_Score,-0.990,-0.994,1.000,0.994,-0.911,0.962,-0.870,0.885,-0.616,0.674
IMD_Rank_of_Average_Rank,-0.984,-0.995,0.994,1.000,-0.882,0.936,-0.839,0.851,-0.570,0.631
IMD_Extent,0.949,0.909,-0.911,-0.882,1.000,-0.961,0.770,-0.822,0.723,-0.765
IMD_Rank_of_Extent,-0.965,-0.944,0.962,0.936,-0.961,1.000,-0.903,0.931,-0.692,0.742
IMD_Local_Concentration,0.829,0.827,-0.870,-0.839,0.770,-0.903,1.000,-0.989,0.626,-0.670
IMD_Rank_of_Local_Concentration,-0.855,-0.843,0.885,0.851,-0.822,0.931,-0.989,1.000,-0.710,0.756
IMD_Proportion_Most_Deprived_10pct,0.643,0.588,-0.616,-0.570,0.723,-0.692,0.626,-0.710,1.000,-0.983
IMD_Rank_of_Proportion_Most_Deprived_10pct,-0.696,-0.645,0.674,0.631,-0.765,0.742,-0.670,0.756,-0.983,1.000


## 12. Feature Selection Review

The correlation and VIF analysis identified several groups of potentially redundant features.

At this stage, no features are removed. The findings are recorded as candidates for the final feature-selection stage, where model performance and data availability will also be considered.

The main areas requiring further review are highly correlated property-price features, population and income variables, and the IMD feature group with limited temporal coverage.

Target variables are kept separate from the predictor features to avoid data leakage.


In [28]:
# Feature selection candidate review

feature_review = pd.DataFrame([
    {
        "Feature_Group": "Property Prices",
        "Features": "Average_Price, Median_Price, Min_Price, Max_Price, Price_STD",
        "Key_Finding": "Price_STD has very high VIF (76.88) and strong correlation with Average_Price (0.921).",
        "Decision_At_This_Stage": "Review for redundancy; do not remove yet."
    },
    {
        "Feature_Group": "Population",
        "Features": "Population, Number_of_Individuals",
        "Key_Finding": "Very strong correlation (0.964). Population also has lower coverage (62.5%).",
        "Decision_At_This_Stage": "Review for redundancy; likely retain the better-covered feature."
    },
    {
        "Feature_Group": "Income",
        "Features": "Mean_Income, Median_Income",
        "Key_Finding": "Strong correlation (0.822). Both have 75% coverage.",
        "Decision_At_This_Stage": "Review for redundancy; compare usefulness during modelling."
    },
    {
        "Feature_Group": "IMD",
        "Features": "All IMD features",
        "Key_Finding": "Only 12.5% coverage and very high correlations between several IMD variables.",
        "Decision_At_This_Stage": "Keep for now; review carefully before modelling."
    },
    {
        "Feature_Group": "Growth Features",
        "Features": "Average_Price_Growth, Median_Price_Growth",
        "Key_Finding": "These variables are closely related to the prediction target and require leakage checking.",
        "Decision_At_This_Stage": "Do not remove yet; verify temporal validity before modelling."
    },
    {
        "Feature_Group": "Target Variables",
        "Features": "Target_Average_Price_Growth, Target_Median_Price_Growth",
        "Key_Finding": "These are future prediction targets.",
        "Decision_At_This_Stage": "Keep completely separate from predictor features."
    }
])

print("=== FEATURE SELECTION CANDIDATE REVIEW ===")
display(feature_review)

=== FEATURE SELECTION CANDIDATE REVIEW ===


,Feature_Group,Features,Key_Finding,Decision_At_This_Stage
0,Property Prices,"Average_Price, Median_Price, Min_Price, Max_Pr...",Price_STD has very high VIF (76.88) and strong...,Review for redundancy; do not remove yet.
1,Population,"Population, Number_of_Individuals",Very strong correlation (0.964). Population al...,Review for redundancy; likely retain the bette...
2,Income,"Mean_Income, Median_Income",Strong correlation (0.822). Both have 75% cove...,Review for redundancy; compare usefulness duri...
3,IMD,All IMD features,Only 12.5% coverage and very high correlations...,Keep for now; review carefully before modelling.
4,Growth Features,"Average_Price_Growth, Median_Price_Growth",These variables are closely related to the pre...,Do not remove yet; verify temporal validity be...
5,Target Variables,"Target_Average_Price_Growth, Target_Median_Pri...",These are future prediction targets.,Keep completely separate from predictor features.


In [31]:
# === TEMPORAL FEATURE VALIDITY CHECK ===

growth_features = [
    "Average_Price_Growth",
    "Median_Price_Growth"
]

target_features = [
    "Target_Average_Price_Growth",
    "Target_Median_Price_Growth"
]

temporal_check = df[
    ["District", "Year"] + growth_features + target_features
].copy()

temporal_check = temporal_check.sort_values(
    ["District", "Year"]
)

print("=== TEMPORAL FEATURE VALIDITY CHECK ===")
display(temporal_check.head(20))

=== TEMPORAL FEATURE VALIDITY CHECK ===


,District,Year,Average_Price_Growth,Median_Price_Growth,Target_Average_Price_Growth,Target_Median_Price_Growth
0,BARKING AND DAGENHAM,2018,NaN,NaN,0.059790,0.010595
1,BARKING AND DAGENHAM,2019,0.059790,0.010595,0.189852,0.032258
2,BARKING AND DAGENHAM,2020,0.189852,0.032258,-0.099888,0.046875
3,BARKING AND DAGENHAM,2021,-0.099888,0.046875,0.192617,0.104478
4,BARKING AND DAGENHAM,2022,0.192617,0.104478,-0.129364,0.010811
5,BARKING AND DAGENHAM,2023,-0.129364,0.010811,-0.070744,-0.037433
6,BARKING AND DAGENHAM,2024,-0.070744,-0.037433,-0.019078,0.055556
7,BARKING AND DAGENHAM,2025,-0.019078,0.055556,NaN,NaN
8,BARNET,2018,NaN,NaN,-0.062592,0.010101
9,BARNET,2019,-0.062592,0.010101,0.048597,0.060000


## 12. Feature Selection Review

The checks above identified several groups of features that may contain redundant information or have limited data coverage.

The main findings are:

* **Property price features:** `Average_Price`, `Median_Price`, `Min_Price`, `Max_Price`, and `Price_STD` contain related information. In particular, `Price_STD` showed a very high VIF (76.88), while `Average_Price` and `Price_STD` also had a strong correlation (0.921). These features should therefore be reviewed for redundancy during model development.
* **Population features:** `Population` and `Number_of_Individuals` are very strongly correlated (0.964). Since `Population` has only 62.5% coverage compared with 75% for `Number_of_Individuals`, their predictive usefulness and data availability should be compared before making a final selection.
* **Income features:** `Mean_Income` and `Median_Income` are strongly correlated (0.822) and both have 75% coverage. Their individual contribution should therefore be compared during modelling rather than automatically retaining both.
* **IMD features:** The IMD variables have only 12.5% coverage and contain several highly correlated measures, including correlations above 0.98. This group requires particular caution because retaining many highly correlated IMD variables may add redundancy without providing substantially different information.
* **Growth features:** `Average_Price_Growth` and `Median_Price_Growth` are related to the historical price dynamics of each borough. Their temporal alignment has been verified, so they are retained as candidate predictor features at this stage.
* **Target variables:** `Target_Average_Price_Growth` and `Target_Median_Price_Growth` represent future outcomes and are kept completely separate from the predictor feature set to prevent data leakage.

At this stage, no features are removed.

The identified redundancies and coverage limitations will be considered during the final feature-selection and modelling stages, where predictive performance, data availability, and model stability can be compared objectively.


## 13. Current Feature Decisions

The feature review does not lead to automatic removal of any variables.

For now:

* `Price_STD` → high-priority redundancy candidate.
* `Population` → redundancy candidate because of its strong overlap with `Number_of_Individuals` and lower coverage.
* `Number_of_Individuals` → stronger coverage than `Population`; keep as a candidate.
* `Mean_Income` and `Median_Income` → correlated features; both remain candidates.
* IMD features → limited coverage and substantial redundancy; treat as a separate candidate group.
* `Average_Price_Growth` and `Median_Price_Growth` → temporally valid historical features; retain as candidates.
* Target variables → never included among predictor features.

Final decisions will be based on model performance and stability rather than correlation or VIF alone.


In [32]:
# === CURRENT PREDICTOR CANDIDATES ===

target_columns = [
    "Target_Average_Price_Growth",
    "Target_Median_Price_Growth"
]

non_feature_columns = [
    "District",
    "Year",
    "Borough_Code"
]

predictor_candidates = [
    col for col in df.columns
    if col not in target_columns + non_feature_columns
]

print("=== CURRENT PREDICTOR CANDIDATES ===")
print("Number of candidates:", len(predictor_candidates))

for i, col in enumerate(predictor_candidates, 1):
    print(f"{i:02d}. {col}")

=== CURRENT PREDICTOR CANDIDATES ===
Number of candidates: 36
01. Transactions
02. Average_Price
03. Median_Price
04. Min_Price
05. Max_Price
06. Price_STD
07. Average_Price_Growth
08. Median_Price_Growth
09. Population
10. Mean_Income
11. Median_Income
12. Number_of_Individuals
13. ARSON AND CRIMINAL DAMAGE
14. BURGLARY
15. DRUG OFFENCES
16. FRAUD AND FORGERY
17. MISCELLANEOUS CRIMES AGAINST SOCIETY
18. POSSESSION OF WEAPONS
19. PUBLIC ORDER OFFENCES
20. ROBBERY
21. SEXUAL OFFENCES
22. THEFT
23. VEHICLE OFFENCES
24. VIOLENCE AGAINST THE PERSON
25. AvPTAI2015
26. PTAL
27. IMD_Average_Rank
28. IMD_Rank_of_Average_Rank
29. IMD_Average_Score
30. IMD_Rank_of_Average_Score
31. IMD_Proportion_Most_Deprived_10pct
32. IMD_Rank_of_Proportion_Most_Deprived_10pct
33. IMD_Extent
34. IMD_Rank_of_Extent
35. IMD_Local_Concentration
36. IMD_Rank_of_Local_Concentration


In [33]:
# ============================================================
# FEATURE SELECTION REVIEW
# ============================================================

# Use the current dataframe
data = df.copy()

# Current predictor candidates
predictor_candidates = [
    col for col in data.columns
    if col not in [
        "Target_Average_Price_Growth",
        "Target_Median_Price_Growth",
        "District",
        "Borough_Code"
    ]
]

# ------------------------------------------------------------
# 1. Coverage
# ------------------------------------------------------------

feature_review = pd.DataFrame({
    "Feature": predictor_candidates,
    "Non_Missing": [
        data[col].notna().sum()
        for col in predictor_candidates
    ],
    "Missing_%": [
        round(data[col].isna().mean() * 100, 1)
        for col in predictor_candidates
    ],
    "Coverage_%": [
        round(data[col].notna().mean() * 100, 1)
        for col in predictor_candidates
    ]
})

# ------------------------------------------------------------
# 2. Flag important feature groups
# ------------------------------------------------------------

def classify_feature(feature):

    if feature in [
        "Average_Price",
        "Median_Price",
        "Min_Price",
        "Max_Price",
        "Price_STD"
    ]:
        return "Property Prices"

    elif feature in [
        "Transactions"
    ]:
        return "Property Activity"

    elif feature in [
        "Average_Price_Growth",
        "Median_Price_Growth"
    ]:
        return "Growth Features"

    elif feature in [
        "Population",
        "Number_of_Individuals"
    ]:
        return "Population"

    elif feature in [
        "Mean_Income",
        "Median_Income"
    ]:
        return "Income"

    elif feature.startswith("IMD_"):
        return "IMD"

    elif feature in [
        "ARSON AND CRIMINAL DAMAGE",
        "BURGLARY",
        "DRUG OFFENCES",
        "FRAUD AND FORGERY",
        "MISCELLANEOUS CRIMES AGAINST SOCIETY",
        "POSSESSION OF WEAPONS",
        "PUBLIC ORDER OFFENCES",
        "ROBBERY",
        "SEXUAL OFFENCES",
        "THEFT",
        "VEHICLE OFFENCES",
        "VIOLENCE AGAINST THE PERSON"
    ]:
        return "Crime"

    elif feature in [
        "AvPTAI2015",
        "PTAL"
    ]:
        return "Transport Accessibility"

    else:
        return "Other"


feature_review["Feature_Group"] = [
    classify_feature(col)
    for col in feature_review["Feature"]
]

# ------------------------------------------------------------
# 3. Decision flags based on what we already discovered
# ------------------------------------------------------------

feature_review["Review_Flag"] = "Keep for now"

feature_review.loc[
    feature_review["Coverage_%"] < 50,
    "Review_Flag"
] = "Low coverage — review"

feature_review.loc[
    feature_review["Feature"] == "Price_STD",
    "Review_Flag"
] = "High redundancy — review"

feature_review.loc[
    feature_review["Feature"].isin([
        "Population",
        "Number_of_Individuals"
    ]),
    "Review_Flag"
] = "Highly correlated pair — review"

feature_review.loc[
    feature_review["Feature"].isin([
        "Mean_Income",
        "Median_Income"
    ]),
    "Review_Flag"
] = "Highly correlated pair — review"

feature_review.loc[
    feature_review["Feature_Group"] == "IMD",
    "Review_Flag"
] = "Very low coverage + high redundancy — review"

# ------------------------------------------------------------
# 4. Sort by feature group and missingness
# ------------------------------------------------------------

feature_review = feature_review.sort_values(
    ["Feature_Group", "Missing_%"],
    ascending=[True, False]
).reset_index(drop=True)

print("=== FEATURE SELECTION REVIEW ===")
display(feature_review)

=== FEATURE SELECTION REVIEW ===


,Feature,Non_Missing,Missing_%,Coverage_%,Feature_Group,Review_Flag
0,ARSON AND CRIMINAL DAMAGE,96,63.6,36.4,Crime,Low coverage — review
1,BURGLARY,96,63.6,36.4,Crime,Low coverage — review
2,DRUG OFFENCES,96,63.6,36.4,Crime,Low coverage — review
3,FRAUD AND FORGERY,96,63.6,36.4,Crime,Low coverage — review
4,MISCELLANEOUS CRIMES AGAINST SOCIETY,96,63.6,36.4,Crime,Low coverage — review
5,POSSESSION OF WEAPONS,96,63.6,36.4,Crime,Low coverage — review
6,PUBLIC ORDER OFFENCES,96,63.6,36.4,Crime,Low coverage — review
7,ROBBERY,96,63.6,36.4,Crime,Low coverage — review
8,SEXUAL OFFENCES,96,63.6,36.4,Crime,Low coverage — review
9,THEFT,96,63.6,36.4,Crime,Low coverage — review


## Feature Selection Review

At this stage, we reviewed the features based on **data coverage, correlation, and potential redundancy**.

No features were removed yet. We only identified the groups that need further review:

* **Crime features:** relatively low coverage
* **IMD features:** very low coverage and high redundancy
* **Price_STD:** high redundancy with other property price features
* **Population / Number_of_Individuals:** highly correlated
* **Mean_Income / Median_Income:** highly correlated

These features will be reviewed before defining the **final predictor set for modelling**.


In [34]:
# Review the coverage pattern of Crime features

crime_features = [
    "ARSON AND CRIMINAL DAMAGE",
    "BURGLARY",
    "DRUG OFFENCES",
    "FRAUD AND FORGERY",
    "MISCELLANEOUS CRIMES AGAINST SOCIETY",
    "POSSESSION OF WEAPONS",
    "PUBLIC ORDER OFFENCES",
    "ROBBERY",
    "SEXUAL OFFENCES",
    "THEFT",
    "VEHICLE OFFENCES",
    "VIOLENCE AGAINST THE PERSON"
]

crime_coverage = pd.DataFrame({
    "Feature": crime_features,
    "Non_Missing": [df[col].notna().sum() for col in crime_features],
    "Missing": [df[col].isna().sum() for col in crime_features],
    "Coverage_%": [
        round(df[col].notna().mean() * 100, 1)
        for col in crime_features
    ]
})

print("=== CRIME FEATURE COVERAGE ===")
display(crime_coverage)

# Check where the crime data exists
crime_available = df[crime_features].notna().any(axis=1)

print("\n=== CRIME DATA BY YEAR ===")
display(
    df.loc[crime_available]
      .groupby("Year")
      .size()
      .reset_index(name="Rows_With_Crime_Data")
)

print("\n=== CRIME DATA BY BOROUGH ===")
display(
    df.loc[crime_available]
      .groupby("District")
      .size()
      .sort_values(ascending=False)
      .reset_index(name="Rows_With_Crime_Data")
)

=== CRIME FEATURE COVERAGE ===


,Feature,Non_Missing,Missing,Coverage_%
0,ARSON AND CRIMINAL DAMAGE,96,168,36.4
1,BURGLARY,96,168,36.4
2,DRUG OFFENCES,96,168,36.4
3,FRAUD AND FORGERY,96,168,36.4
4,MISCELLANEOUS CRIMES AGAINST SOCIETY,96,168,36.4
5,POSSESSION OF WEAPONS,96,168,36.4
6,PUBLIC ORDER OFFENCES,96,168,36.4
7,ROBBERY,96,168,36.4
8,SEXUAL OFFENCES,96,168,36.4
9,THEFT,96,168,36.4



=== CRIME DATA BY YEAR ===


,Year,Rows_With_Crime_Data
0,2021,32
1,2022,32
2,2023,32



=== CRIME DATA BY BOROUGH ===


,District,Rows_With_Crime_Data
0,BARKING AND DAGENHAM,3
1,BARNET,3
2,BEXLEY,3
3,BRENT,3
4,BROMLEY,3
5,CAMDEN,3
6,CITY OF WESTMINSTER,3
7,CROYDON,3
8,EALING,3
9,ENFIELD,3


In [35]:
crime_available = df[crime_features].notna().any(axis=1)

print("=== CRIME DATA BY YEAR ===")
display(
    df.loc[crime_available]
      .groupby("Year")
      .size()
      .reset_index(name="Rows_With_Crime_Data")
)

print("\n=== CRIME DATA BY BOROUGH ===")
display(
    df.loc[crime_available]
      .groupby("District")
      .size()
      .sort_values(ascending=False)
      .reset_index(name="Rows_With_Crime_Data")
)

=== CRIME DATA BY YEAR ===


,Year,Rows_With_Crime_Data
0,2021,32
1,2022,32
2,2023,32



=== CRIME DATA BY BOROUGH ===


,District,Rows_With_Crime_Data
0,BARKING AND DAGENHAM,3
1,BARNET,3
2,BEXLEY,3
3,BRENT,3
4,BROMLEY,3
5,CAMDEN,3
6,CITY OF WESTMINSTER,3
7,CROYDON,3
8,EALING,3
9,ENFIELD,3


In [37]:
imd_available = df[imd_features].notna().any(axis=1)

print("=== IMD DATA BY YEAR ===")
display(
    df.loc[imd_available]
    .groupby("Year")
    .size()
    .reset_index(name="Rows_With_IMD_Data")
)

print("\n=== IMD DATA BY BOROUGH ===")
display(
    df.loc[imd_available]
    .groupby("District")
    .size()
    .sort_values(ascending=False)
    .reset_index(name="Rows_With_IMD_Data")
)

=== IMD DATA BY YEAR ===


,Year,Rows_With_IMD_Data
0,2019,33



=== IMD DATA BY BOROUGH ===


,District,Rows_With_IMD_Data
0,BARKING AND DAGENHAM,1
1,BARNET,1
2,BEXLEY,1
3,BRENT,1
4,BROMLEY,1
5,CAMDEN,1
6,CITY OF LONDON,1
7,CITY OF WESTMINSTER,1
8,CROYDON,1
9,EALING,1


In [38]:
population_features = [
    "Population",
    "Number_of_Individuals"
]

population_review = pd.DataFrame({
    "Feature": population_features,
    "Non_Missing": [df[col].notna().sum() for col in population_features],
    "Missing": [df[col].isna().sum() for col in population_features],
    "Coverage_%": [
        round(df[col].notna().mean() * 100, 1)
        for col in population_features
    ],
    "Mean": [df[col].mean() for col in population_features],
    "Median": [df[col].median() for col in population_features]
})

print("=== POPULATION FEATURE REVIEW ===")
display(population_review)

print(
    "\nCorrelation:",
    df["Population"].corr(df["Number_of_Individuals"])
)

=== POPULATION FEATURE REVIEW ===


,Feature,Non_Missing,Missing,Coverage_%,Mean,Median
0,Population,165,99,62.5,268266.048485,279527.0
1,Number_of_Individuals,198,66,75.0,133217.171717,136000.0



Correlation: 0.9638707195654206


In [39]:
income_features = [
    "Mean_Income",
    "Median_Income"
]

income_review = pd.DataFrame({
    "Feature": income_features,
    "Non_Missing": [df[col].notna().sum() for col in income_features],
    "Missing": [df[col].isna().sum() for col in income_features],
    "Coverage_%": [
        round(df[col].notna().mean() * 100, 1)
        for col in income_features
    ],
    "Mean": [df[col].mean() for col in income_features],
    "Median": [df[col].median() for col in income_features]
})

print("=== INCOME FEATURE REVIEW ===")
display(income_review)

print(
    "\nCorrelation:",
    df["Mean_Income"].corr(df["Median_Income"])
)

=== INCOME FEATURE REVIEW ===


,Feature,Non_Missing,Missing,Coverage_%,Mean,Median
0,Mean_Income,198,66,75.0,61880.808081,49000.0
1,Median_Income,198,66,75.0,34145.959596,32650.0



Correlation: 0.8216443079945169


In [40]:
price_features = [
    "Average_Price",
    "Median_Price",
    "Min_Price",
    "Max_Price",
    "Price_STD"
]

price_review = pd.DataFrame({
    "Feature": price_features,
    "Non_Missing": [df[col].notna().sum() for col in price_features],
    "Missing": [df[col].isna().sum() for col in price_features],
    "Coverage_%": [
        round(df[col].notna().mean() * 100, 1)
        for col in price_features
    ],
    "Mean": [df[col].mean() for col in price_features],
    "Median": [df[col].median() for col in price_features]
})

print("=== PROPERTY PRICE FEATURE REVIEW ===")
display(price_review)

print("\n=== PRICE FEATURE CORRELATION ===")
display(
    df[price_features]
    .corr()
    .round(3)
)

=== PROPERTY PRICE FEATURE REVIEW ===


,Feature,Non_Missing,Missing,Coverage_%,Mean,Median
0,Average_Price,264,0,100.0,9.698969e+05,6.951213e+05
1,Median_Price,264,0,100.0,5.530846e+05,5.000000e+05
2,Min_Price,264,0,100.0,8.817197e+02,5.000000e+02
3,Max_Price,264,0,100.0,9.403684e+07,5.500000e+07
4,Price_STD,264,0,100.0,2.929790e+06,1.537734e+06



=== PRICE FEATURE CORRELATION ===


,Average_Price,Median_Price,Min_Price,Max_Price,Price_STD
Average_Price,1.000,0.649,0.300,0.429,0.921
Median_Price,0.649,1.000,0.198,0.387,0.536
Min_Price,0.300,0.198,1.000,0.002,0.252
Max_Price,0.429,0.387,0.002,1.000,0.702
Price_STD,0.921,0.536,0.252,0.702,1.000


In [42]:
print("=== TRANSPORT FEATURE DATA TYPES ===")
print(df[["AvPTAI2015", "PTAL"]].dtypes)

print("\n=== PTAL SAMPLE VALUES ===")
display(df["PTAL"].dropna().head(20))

print("\n=== PTAL UNIQUE VALUE COUNT ===")
print(df["PTAL"].nunique(dropna=True))

=== TRANSPORT FEATURE DATA TYPES ===
AvPTAI2015    float64
PTAL           object
dtype: object

=== PTAL SAMPLE VALUES ===


0      2
1      2
2      2
3      2
4      2
5      2
6      2
7      2
8      2
9      2
10     2
11     2
12     2
13     2
14     2
15     2
16    1b
17    1b
18    1b
19    1b
Name: PTAL, dtype: object


=== PTAL UNIQUE VALUE COUNT ===
7


In [43]:
print("=== PTAL VALUE COUNTS ===")
display(df["PTAL"].value_counts(dropna=False).sort_index())

print("\n=== TRANSPORT FEATURE COVERAGE ===")
transport_coverage = pd.DataFrame({
    "Feature": ["AvPTAI2015", "PTAL"],
    "Non_Missing": [
        df["AvPTAI2015"].notna().sum(),
        df["PTAL"].notna().sum()
    ],
    "Missing": [
        df["AvPTAI2015"].isna().sum(),
        df["PTAL"].isna().sum()
    ],
    "Coverage_%": [
        round(df["AvPTAI2015"].notna().mean() * 100, 1),
        round(df["PTAL"].notna().mean() * 100, 1)
    ]
})

display(transport_coverage)

print("\n=== AvPTAI2015 SUMMARY ===")
display(
    df["AvPTAI2015"].describe()
)

=== PTAL VALUE COUNTS ===


PTAL
1b     48
2     104
3      32
4      24
5      32
6a     16
6b      8
Name: count, dtype: int64


=== TRANSPORT FEATURE COVERAGE ===


,Feature,Non_Missing,Missing,Coverage_%
0,AvPTAI2015,264,0,100.0
1,PTAL,264,0,100.0



=== AvPTAI2015 SUMMARY ===


count    264.000000
mean      14.022116
std       15.961429
min        2.747859
25%        5.542330
50%        8.503510
75%       17.478762
max       90.802508
Name: AvPTAI2015, dtype: float64

In [44]:
# Check how AvPTAI2015 varies across PTAL levels

ptal_order = ["1b", "2", "3", "4", "5", "6a", "6b"]

transport_by_ptal = (
    df.groupby("PTAL")["AvPTAI2015"]
    .agg(["count", "mean", "median", "min", "max"])
    .reindex(ptal_order)
)

print("=== AvPTAI2015 BY PTAL LEVEL ===")
display(transport_by_ptal)

print("\n=== SPEARMAN CORRELATION ===")

ptal_numeric = df["PTAL"].map({
    "1b": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6a": 6,
    "6b": 7
})

print(
    df["AvPTAI2015"].corr(ptal_numeric, method="spearman")
)

=== AvPTAI2015 BY PTAL LEVEL ===


,count,mean,median,min,max
PTAL,,,,,
1b,48,3.996402,4.072506,2.747859,4.896334
2,104,6.815542,5.842208,5.396992,9.223249
3,32,12.564407,12.602602,11.079336,13.973088
4,24,17.445868,17.478762,16.325973,18.532870
5,32,22.583317,22.751302,20.513884,24.316782
6a,16,33.209181,33.209181,26.667566,39.750796
6b,8,90.802508,90.802508,90.802508,90.802508



=== SPEARMAN CORRELATION ===
0.9639213583206557


In [45]:
# Check temporal relationship between current growth features and next-year targets

growth_features = [
    "Average_Price_Growth",
    "Median_Price_Growth"
]

target_features = [
    "Target_Average_Price_Growth",
    "Target_Median_Price_Growth"
]

temporal_check = df[
    ["District", "Year"] + growth_features + target_features
].copy()

temporal_check = temporal_check.sort_values(
    ["District", "Year"]
)

print("=== TEMPORAL VALIDITY CHECK ===")
display(temporal_check.head(20))

=== TEMPORAL VALIDITY CHECK ===


,District,Year,Average_Price_Growth,Median_Price_Growth,Target_Average_Price_Growth,Target_Median_Price_Growth
0,BARKING AND DAGENHAM,2018,NaN,NaN,0.059790,0.010595
1,BARKING AND DAGENHAM,2019,0.059790,0.010595,0.189852,0.032258
2,BARKING AND DAGENHAM,2020,0.189852,0.032258,-0.099888,0.046875
3,BARKING AND DAGENHAM,2021,-0.099888,0.046875,0.192617,0.104478
4,BARKING AND DAGENHAM,2022,0.192617,0.104478,-0.129364,0.010811
5,BARKING AND DAGENHAM,2023,-0.129364,0.010811,-0.070744,-0.037433
6,BARKING AND DAGENHAM,2024,-0.070744,-0.037433,-0.019078,0.055556
7,BARKING AND DAGENHAM,2025,-0.019078,0.055556,NaN,NaN
8,BARNET,2018,NaN,NaN,-0.062592,0.010101
9,BARNET,2019,-0.062592,0.010101,0.048597,0.060000


In [46]:
print("=== GROWTH FEATURE vs NEXT-YEAR TARGET ===")

print(
    "Average growth → Target average growth:",
    df["Average_Price_Growth"].corr(
        df["Target_Average_Price_Growth"]
    )
)

print(
    "Median growth → Target median growth:",
    df["Median_Price_Growth"].corr(
        df["Target_Median_Price_Growth"]
    )
)

=== GROWTH FEATURE vs NEXT-YEAR TARGET ===
Average growth → Target average growth: -0.5412100188644774
Median growth → Target median growth: -0.04246138421491539


## Feature Selection Summary

We reviewed the main predictor groups based on data coverage, redundancy, multicollinearity, and temporal validity.

At this stage:

- Crime features were removed due to limited coverage.
- IMD features were removed due to very low coverage and high redundancy.
- `Population` and `Mean_Income` were removed due to redundancy with other features.
- `Price_STD` was removed due to very high VIF and strong redundancy with `Average_Price`.
- `PTAL` was removed because it provides highly similar information to `AvPTAI2015`.
- `Average_Price_Growth` and `Median_Price_Growth` were retained after confirming their temporal alignment with the next-year targets.

The remaining features will now be checked again for multicollinearity before defining the final predictor set.